# NIH ChestX-ray14 — Multi-Label Disease Classification with DenseNet121 & Grad-CAM

This notebook covers:
1. **Data Loading** — NIH ChestX-ray14 dataset (~112k images, 14 pathologies)
2. **Preprocessing** — Multi-label encoding, train/val/test split, transforms
3. **Model** — DenseNet121 fine-tuned for multi-label classification
4. **Training** — BCE loss, mixed-precision, AUC metrics
5. **Grad-CAM** — Heatmap visualization for explainability
6. **Inference** — Predict + overlay heatmap on X-ray

## 0. Setup & Imports

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.amp import GradScaler, autocast

import torchvision.transforms as T
import torchvision.models as models

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

import cv2

# Reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# Check GPU compatibility — P100 (sm_60) is not supported by recent PyTorch
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability()
    if cap[0] < 7:
        print(f"WARNING: GPU {torch.cuda.get_device_name()} (sm_{cap[0]}{cap[1]}) is not supported by this PyTorch build.")
        print("Switch to T4 GPU in Kaggle: Settings → Accelerator → GPU T4 x2")
        print("Falling back to CPU.\n")
        DEVICE = torch.device("cpu")
    else:
        DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")
print(f"Using device: {DEVICE}")

## 1. Configuration

In [ ]:
# === PATHS (Kaggle dataset layout) ===
# On Kaggle, add the dataset "nih-chest-xrays/data" as input.
# Auto-detect the actual base directory (path varies by Kaggle environment)
_CANDIDATES = [
    "/kaggle/input/data",
    "/kaggle/input/datasets/organizations/nih-chest-xrays/data",
    "/kaggle/input/nih-chest-xrays-data",
]
BASE_DIR = None
for _c in _CANDIDATES:
    if os.path.isdir(_c):
        BASE_DIR = _c
        break

if BASE_DIR is None:
    # Fallback: search /kaggle/input recursively for Data_Entry_2017.csv
    for root, dirs, files in os.walk("/kaggle/input"):
        if "Data_Entry_2017.csv" in files:
            BASE_DIR = root
            break

assert BASE_DIR is not None, (
    "Could not find dataset! Make sure 'nih-chest-xrays/data' is added as a Kaggle input dataset."
)
print(f"Dataset found at: {BASE_DIR}")

CSV_PATH = os.path.join(BASE_DIR, "Data_Entry_2017.csv")

# Collect all image directories
IMAGE_DIRS = []
for folder in sorted(os.listdir(BASE_DIR)):
    full = os.path.join(BASE_DIR, folder)
    if not os.path.isdir(full):
        continue
    # Layout A: images_00X/images/*.png
    sub = os.path.join(full, "images")
    if os.path.isdir(sub):
        IMAGE_DIRS.append(sub)
    # Layout B: images_00X/*.png  (flat)
    elif folder.startswith("images_"):
        IMAGE_DIRS.append(full)

print(f"Found {len(IMAGE_DIRS)} image directories")
for d in IMAGE_DIRS:
    print(f"  {d}  ({len(os.listdir(d))} files)")

In [ ]:
# === HYPERPARAMETERS ===
PATHOLOGIES = [
    "Atelectasis", "Cardiomegaly", "Effusion", "Infiltration",
    "Mass", "Nodule", "Pneumonia", "Pneumothorax",
    "Consolidation", "Edema", "Emphysema", "Fibrosis",
    "Pleural_Thickening", "Hernia"
]
NUM_CLASSES = len(PATHOLOGIES)

IMG_SIZE = 224
BATCH_SIZE = 128          # doubled — fits on P100/T4 (16GB)
NUM_WORKERS = 2           # use 0 if you see multiprocessing AssertionErrors
LEARNING_RATE = 1e-4
NUM_EPOCHS = 5            # 5 is usually enough to converge
VAL_RATIO = 0.1
TEST_RATIO = 0.1

print(f"Pathologies ({NUM_CLASSES}): {PATHOLOGIES}")

## 2. Data Loading & Multi-Label Encoding

In [ ]:
# Load the metadata CSV
df = pd.read_csv(CSV_PATH)
print(f"Total entries: {len(df)}")
df.head()

In [ ]:
# Build a fast lookup: filename -> full path
filename_to_path = {}
for img_dir in IMAGE_DIRS:
    for fname in os.listdir(img_dir):
        if fname.endswith(".png"):
            filename_to_path[fname] = os.path.join(img_dir, fname)

print(f"Indexed {len(filename_to_path)} images on disk")

# Map each row to its full path
df["full_path"] = df["Image Index"].map(filename_to_path)
# Drop rows where the image file is missing
before = len(df)
df = df.dropna(subset=["full_path"]).reset_index(drop=True)
print(f"Matched {len(df)} / {before} entries to image files")

In [ ]:
# Multi-label encoding: "Finding Labels" column contains pipe-separated labels
# e.g. "Mass|Effusion" or "No Finding"
for pathology in PATHOLOGIES:
    df[pathology] = df["Finding Labels"].apply(lambda x: 1.0 if pathology in x.split("|") else 0.0)

# Quick distribution check
print("\nLabel distribution:")
print(df[PATHOLOGIES].sum().sort_values(ascending=False))
print(f"\nNo Finding count: {(df['Finding Labels'] == 'No Finding').sum()}")

In [ ]:
# Patient-level split to prevent data leakage
# (same patient should not appear in both train and val/test)
patient_ids = df["Patient ID"].unique()
train_patients, temp_patients = train_test_split(
    patient_ids, test_size=VAL_RATIO + TEST_RATIO, random_state=SEED
)
val_patients, test_patients = train_test_split(
    temp_patients, test_size=TEST_RATIO / (VAL_RATIO + TEST_RATIO), random_state=SEED
)

train_df = df[df["Patient ID"].isin(train_patients)].reset_index(drop=True)
val_df   = df[df["Patient ID"].isin(val_patients)].reset_index(drop=True)
test_df  = df[df["Patient ID"].isin(test_patients)].reset_index(drop=True)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

## 3. Dataset & DataLoader

In [ ]:
class ChestXrayDataset(Dataset):
    """NIH ChestX-ray14 multi-label dataset."""

    def __init__(self, dataframe, pathologies, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.pathologies = pathologies
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["full_path"]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        label = torch.tensor(row[self.pathologies].values.astype(np.float32))
        return img, label

In [ ]:
# Transforms
train_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(),
    T.RandomRotation(10),
    T.ColorJitter(brightness=0.1, contrast=0.1),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]),
])

val_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]),
])

# Datasets
train_ds = ChestXrayDataset(train_df, PATHOLOGIES, transform=train_transform)
val_ds   = ChestXrayDataset(val_df,   PATHOLOGIES, transform=val_transform)
test_ds  = ChestXrayDataset(test_df,  PATHOLOGIES, transform=val_transform)

# DataLoaders
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)} | Test batches: {len(test_loader)}")

In [ ]:
# Sanity check: visualise a batch
imgs, labels = next(iter(train_loader))
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for i, ax in enumerate(axes):
    img_np = imgs[i].permute(1, 2, 0).numpy()
    img_np = img_np * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
    img_np = np.clip(img_np, 0, 1)
    ax.imshow(img_np)
    active = [PATHOLOGIES[j] for j in range(NUM_CLASSES) if labels[i, j] == 1.0]
    ax.set_title(", ".join(active) if active else "No Finding", fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 4. DenseNet121 Model

In [ ]:
class DenseNet121MultiLabel(nn.Module):
    def __init__(self, num_classes, pretrained=True):
        super().__init__()
        self.densenet = models.densenet121(
            weights=models.DenseNet121_Weights.IMAGENET1K_V1 if pretrained else None
        )
        in_features = self.densenet.classifier.in_features
        self.densenet.classifier = nn.Sequential(
            nn.Linear(in_features, num_classes),
        )

    def forward(self, x):
        # Manual forward to avoid inplace ReLU (breaks Grad-CAM backward hooks)
        features = self.densenet.features(x)
        out = torch.relu(features)  # NOT inplace
        out = nn.functional.adaptive_avg_pool2d(out, (1, 1))
        out = torch.flatten(out, 1)
        out = self.densenet.classifier(out)
        return out

model = DenseNet121MultiLabel(NUM_CLASSES, pretrained=True).to(DEVICE)

# Freeze early layers (denseblock1 & denseblock2) — speeds up training ~2x
# Only fine-tune denseblock3, denseblock4, and the classifier
for name, param in model.densenet.features.named_parameters():
    if "denseblock3" in name or "denseblock4" in name or "norm5" in name:
        param.requires_grad = True
    else:
        param.requires_grad = False

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Total params: {total:,} | Trainable: {trainable:,} ({100*trainable/total:.1f}%)")

## 5. Training Loop

In [ ]:
class FocalLoss(nn.Module):
    """Focal Loss for multi-label classification.
    
    Reduces the loss contribution from easy/common negatives and focuses
    training on hard, misclassified examples (e.g., rare pathologies).
    
    FL(p_t) = -alpha_t * (1 - p_t)^gamma * log(p_t)
    
    Args:
        alpha: weighting factor for positives (balances pos/neg). Default 0.25.
        gamma: focusing parameter. Higher = more focus on hard examples. Default 2.0.
    """
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        bce = nn.functional.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        probs = torch.sigmoid(logits)
        p_t = probs * targets + (1 - probs) * (1 - targets)
        alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        focal_weight = alpha_t * (1 - p_t) ** self.gamma
        loss = focal_weight * bce
        return loss.mean()

print("Focal Loss defined (alpha=0.25, gamma=2.0)")

In [ ]:
criterion = FocalLoss(alpha=0.25, gamma=2.0)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)
scaler = GradScaler()

In [ ]:
def compute_auc(y_true, y_scores):
    """Compute per-class AUC, skipping classes with no positive samples."""
    aucs = []
    for i in range(y_true.shape[1]):
        if y_true[:, i].sum() == 0:
            continue
        try:
            auc = roc_auc_score(y_true[:, i], y_scores[:, i])
            aucs.append(auc)
        except ValueError:
            continue
    return np.mean(aucs) if aucs else 0.0

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, scaler):
    model.train()
    running_loss = 0.0
    all_labels, all_preds = [], []

    for imgs, labels in tqdm(loader, desc="Training", leave=False):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()
        with autocast("cuda"):
            logits = model(imgs)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * imgs.size(0)
        all_labels.append(labels.cpu().numpy())
        all_preds.append(torch.sigmoid(logits).detach().cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    all_labels = np.concatenate(all_labels)
    all_preds = np.concatenate(all_preds)
    epoch_auc = compute_auc(all_labels, all_preds)
    return epoch_loss, epoch_auc


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    running_loss = 0.0
    all_labels, all_preds = [], []

    for imgs, labels in tqdm(loader, desc="Evaluating", leave=False):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        with autocast("cuda"):
            logits = model(imgs)
            loss = criterion(logits, labels)

        running_loss += loss.item() * imgs.size(0)
        all_labels.append(labels.cpu().numpy())
        all_preds.append(torch.sigmoid(logits).cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    all_labels = np.concatenate(all_labels)
    all_preds = np.concatenate(all_preds)
    epoch_auc = compute_auc(all_labels, all_preds)
    return epoch_loss, epoch_auc

In [ ]:
best_val_auc = 0.0
history = {"train_loss": [], "val_loss": [], "train_auc": [], "val_auc": []}

for epoch in range(1, NUM_EPOCHS + 1):
    print(f"\n{'='*50}")
    print(f"Epoch {epoch}/{NUM_EPOCHS}")

    train_loss, train_auc = train_one_epoch(model, train_loader, criterion, optimizer, scaler)
    val_loss, val_auc = evaluate(model, val_loader, criterion)

    scheduler.step(val_auc)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_auc"].append(train_auc)
    history["val_auc"].append(val_auc)

    print(f"  Train Loss: {train_loss:.4f} | Train AUC: {train_auc:.4f}")
    print(f"  Val   Loss: {val_loss:.4f}   | Val   AUC: {val_auc:.4f}")

    # Save best model
    if val_auc > best_val_auc:
        best_val_auc = val_auc
        torch.save(model.state_dict(), "best_densenet121_nih.pth")
        print(f"  >> Saved best model (AUC={val_auc:.4f})")

print(f"\nBest validation AUC: {best_val_auc:.4f}")

In [ ]:
# Training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history["train_loss"], label="Train Loss")
ax1.plot(history["val_loss"], label="Val Loss")
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss"); ax1.legend(); ax1.set_title("Loss")

ax2.plot(history["train_auc"], label="Train AUC")
ax2.plot(history["val_auc"], label="Val AUC")
ax2.set_xlabel("Epoch"); ax2.set_ylabel("AUC"); ax2.legend(); ax2.set_title("Mean AUC")

plt.tight_layout()
plt.show()

## 6. Test Set Evaluation (Per-Class AUC)

In [ ]:
# Load best model
model.load_state_dict(torch.load("best_densenet121_nih.pth", map_location=DEVICE, weights_only=True))
model.eval()

all_labels, all_preds = [], []
with torch.no_grad():
    for imgs, labels in tqdm(test_loader, desc="Testing"):
        imgs = imgs.to(DEVICE)
        with autocast("cuda"):
            logits = model(imgs)
        all_labels.append(labels.numpy())
        all_preds.append(torch.sigmoid(logits).cpu().numpy())

all_labels = np.concatenate(all_labels)
all_preds = np.concatenate(all_preds)

print("\nPer-class AUC on TEST set:")
print("-" * 35)
per_class_auc = {}
for i, pathology in enumerate(PATHOLOGIES):
    if all_labels[:, i].sum() == 0:
        print(f"  {pathology:25s}  N/A (no positives)")
        continue
    auc = roc_auc_score(all_labels[:, i], all_preds[:, i])
    per_class_auc[pathology] = auc
    print(f"  {pathology:25s}  {auc:.4f}")

mean_auc = np.mean(list(per_class_auc.values()))
print(f"\n  {'Mean AUC':25s}  {mean_auc:.4f}")

## 7. Grad-CAM Implementation

In [ ]:
class GradCAM:
    """Grad-CAM for DenseNet121.
    
    Hooks into the last convolutional layer (densenet.features)
    and computes class-specific heatmaps.
    """

    def __init__(self, model):
        self.model = model
        self.model.eval()
        self.gradients = None
        self.activations = None

        # Target layer: last conv block in DenseNet features
        target_layer = self.model.densenet.features
        target_layer.register_forward_hook(self._save_activation)
        target_layer.register_full_backward_hook(self._save_gradient)

    def _save_activation(self, module, input, output):
        self.activations = output.clone().detach()

    def _save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def generate(self, input_tensor, class_idx):
        """Generate Grad-CAM heatmap for a given class index.
        
        Args:
            input_tensor: (1, 3, H, W) tensor on DEVICE
            class_idx: int, index into PATHOLOGIES
        Returns:
            heatmap: (H, W) numpy array in [0, 1]
        """
        self.model.zero_grad()
        logits = self.model(input_tensor)          # (1, num_classes)
        target = logits[0, class_idx]
        target.backward()

        # Global average pool the gradients -> channel weights
        weights = self.gradients.mean(dim=(2, 3), keepdim=True)  # (1, C, 1, 1)
        cam = (weights * self.activations).sum(dim=1, keepdim=True)  # (1, 1, h, w)
        cam = torch.relu(cam)
        cam = cam.squeeze().cpu().numpy()

        # Normalize to [0, 1]
        if cam.max() > 0:
            cam = cam / cam.max()

        # Resize to input image size
        cam = cv2.resize(cam, (IMG_SIZE, IMG_SIZE))
        return cam

In [ ]:
def overlay_heatmap(image_np, heatmap, alpha=0.4):
    """Overlay a Grad-CAM heatmap on an image.
    
    Args:
        image_np: (H, W, 3) numpy array in [0, 1]
        heatmap:  (H, W) numpy array in [0, 1]
        alpha: blending strength
    Returns:
        blended: (H, W, 3) numpy array in [0, 1]
    """
    colormap = cv2.applyColorMap(np.uint8(255 * heatmap), cv2.COLORMAP_JET)
    colormap = cv2.cvtColor(colormap, cv2.COLOR_BGR2RGB) / 255.0
    blended = alpha * colormap + (1 - alpha) * image_np
    blended = np.clip(blended, 0, 1)
    return blended

## 8. Inference with Grad-CAM Visualization

In [ ]:
def predict_and_explain(model, image_path, grad_cam, top_k=3):
    """Run inference on a single X-ray and produce Grad-CAM heatmaps.
    
    Returns:
        results: list of dicts with keys: pathology, confidence, heatmap
        image_np: the original image as numpy for display
    """
    # Load and preprocess
    orig_img = Image.open(image_path).convert("RGB")
    img_resized = orig_img.resize((IMG_SIZE, IMG_SIZE))
    image_np = np.array(img_resized) / 255.0

    input_tensor = val_transform(orig_img).unsqueeze(0).to(DEVICE)
    input_tensor.requires_grad_(True)

    # Forward pass
    with torch.no_grad():
        logits = model(input_tensor)
    probs = torch.sigmoid(logits).squeeze().cpu().numpy()

    # Get top-k predictions
    top_indices = np.argsort(probs)[::-1][:top_k]

    results = []
    for idx in top_indices:
        # Need a fresh forward pass with gradients for Grad-CAM
        inp = val_transform(orig_img).unsqueeze(0).to(DEVICE)
        inp.requires_grad_(True)
        heatmap = grad_cam.generate(inp, int(idx))
        results.append({
            "pathology": PATHOLOGIES[idx],
            "confidence": float(probs[idx]),
            "heatmap": heatmap,
        })

    return results, image_np

In [ ]:
# Initialize Grad-CAM
grad_cam = GradCAM(model)

# Pick sample images from the test set (ones with actual findings)
positive_samples = test_df[test_df["Finding Labels"] != "No Finding"].sample(n=4, random_state=SEED)

for _, row in positive_samples.iterrows():
    img_path = row["full_path"]
    ground_truth = row["Finding Labels"]

    results, image_np = predict_and_explain(model, img_path, grad_cam, top_k=3)

    # Display
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))

    # Original image
    axes[0].imshow(image_np)
    axes[0].set_title(f"Ground Truth:\n{ground_truth}", fontsize=10)
    axes[0].axis("off")

    # Grad-CAM for top 3 predictions
    for i, res in enumerate(results):
        blended = overlay_heatmap(image_np, res["heatmap"])
        axes[i + 1].imshow(blended)
        axes[i + 1].set_title(
            f"{res['pathology']}\n{res['confidence']*100:.1f}%",
            fontsize=10,
            color="red" if res["confidence"] > 0.5 else "orange"
        )
        axes[i + 1].axis("off")

    plt.suptitle(f"Image: {os.path.basename(img_path)}", fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.show()
    print()

## 9. Export Utilities (for Multi-Agent Pipeline)

In [ ]:
def run_radiomics_tool(image_path, model, grad_cam, top_k=3):
    """Simulates the Radiomics Specialist agent's PyTorch tool.
    
    Returns a structured dict ready for downstream agents:
    - predictions: top-k pathologies with confidence
    - heatmap_paths: saved heatmap overlay images
    """
    results, image_np = predict_and_explain(model, image_path, grad_cam, top_k=top_k)

    output = {
        "source_image": image_path,
        "predictions": [],
        "heatmap_paths": [],
    }

    os.makedirs("gradcam_outputs", exist_ok=True)

    for res in results:
        output["predictions"].append({
            "pathology": res["pathology"],
            "confidence": round(res["confidence"] * 100, 2),
        })

        # Save heatmap overlay
        blended = overlay_heatmap(image_np, res["heatmap"])
        fname = f"gradcam_outputs/{os.path.basename(image_path).replace('.png', '')}_{res['pathology']}.png"
        plt.imsave(fname, blended)
        output["heatmap_paths"].append(fname)

    return output


# Demo: run the tool on a sample image
sample_path = positive_samples.iloc[0]["full_path"]
tool_output = run_radiomics_tool(sample_path, model, grad_cam)

print("=== Radiomics Specialist Output ===")
print(f"Source: {tool_output['source_image']}")
for pred in tool_output["predictions"]:
    print(f"  {pred['pathology']}: {pred['confidence']}%")
print(f"Heatmaps saved: {tool_output['heatmap_paths']}")

## 10. Save Model for Later Use

In [ ]:
# Save the full checkpoint for downstream use
checkpoint = {
    "model_state_dict": model.state_dict(),
    "pathologies": PATHOLOGIES,
    "num_classes": NUM_CLASSES,
    "img_size": IMG_SIZE,
    "per_class_auc": per_class_auc,
    "mean_auc": mean_auc,
}
torch.save(checkpoint, "densenet121_nih_checkpoint.pth")
print("Checkpoint saved to densenet121_nih_checkpoint.pth")
print(f"Final Mean AUC: {mean_auc:.4f}")